In [1]:
import os
import pandas as pd
from slugify import slugify
import numpy as np
from corrosions import AcvgDcvg, PCM, CIPS

In [2]:
acvg_dir = r"D:\Data\ACVG"
cips_dir = r"D:\Data\CIPS"
pcm_dir = r"D:\Data\PCM"

In [3]:
files_excel = r"D:\Projects\corrosions\tests\files.xlsx"

In [4]:
def get_code(row):
    code = f"{row['Segment']} {row['Diameter']}"
    return slugify(code)

In [5]:
def get_area_code(row):
    area_code = f"{row['Area']} {row['Year']}"
    return slugify(area_code)

In [6]:
df = pd.read_excel(files_excel)

In [7]:
df['code'] = df.apply(lambda row: get_code(row), axis=1)
df['area_code'] = df.apply(lambda row: get_area_code(row), axis=1)

In [8]:
df.rename(columns={
    'Year': 'year',
    'Area': 'area',
    'Nomor Segment': 'nomor',
    'Segment': 'name',
    'Diameter': 'diameter',
    'Length': 'pipe_length',
    'Province Code': 'province_code',
    'Note': 'note'
}, inplace=True)

In [9]:
to_process = []

for idx in df.index:
    row = df.loc[idx]

    acvg_file = None
    cips_file = None
    pcm_file = None

    if row['CIPS'] is np.nan or row['PCM'] is np.nan:
        print(f"{idx}: PCM or CIPS file does not exist")
        print(f"    PCM : {row['PCM']}")
        print(f"    CIPS : {row['CIPS']}")
        continue

    if row['ACVG/DCVG'] is not np.nan:
        acvg_file = os.path.join(acvg_dir, row['ACVG/DCVG'])
        if not os.path.exists(acvg_file):
            print(f"{acvg_file} does not exist")
        else:
            acvg_file = AcvgDcvg(
                file_or_dir=acvg_file,
                segment_code=row['code'],
                overwrite=True,
            )

    if row['CIPS'] is not np.nan:
        cips_file = os.path.join(cips_dir, row['CIPS'])
        if not os.path.exists(cips_file):
            print(f"{cips_file} does not exist")
        else:
            cips_file = CIPS(
                file_or_dir=cips_file,
            )

    if row['PCM'] is not np.nan:
        pcm_file = os.path.join(pcm_dir, row['PCM'])
        if not os.path.exists(pcm_file):
            print(f"{pcm_file} does not exist")
        else:
            pcm_file = PCM(
                file_or_dir=pcm_file,
            )

    _process = {
        'id': idx,
        'year': row['year'],
        'area' : row['area'],
        'area_code' : row['area_code'],
        'nomor' : row['nomor'],
        'segment': row['name'],
        'diameter': row['diameter'],
        'length': row['pipe_length'],
        'code': row['code'],
        'province_code': row['province_code'],
        'note': row['note'],
        'acvg_dcvg': acvg_file,
        'cips': cips_file,
        'pcm': pcm_file,
        'file_acvg_dcvg': row['ACVG/DCVG'],
        'file_cips': row['CIPS'],
        'file_pcm': row['PCM'],
    }

    to_process.append(_process)

In [10]:
results = []

for process in to_process:
    # print(process['id'], process['file_pcm'])

    if process['acvg_dcvg'] is not None:
        process['acvg_dcvg'].normalize()

    if process['cips'] is None or process['pcm'] is None:
        continue

    process['cips'].normalize()
    process['pcm'].normalize()

    _result = {
        'id': process['id'],
        'year': process['year'],
        'area': process['area'],
        'area_code': process['area_code'],
        'nomor': process['nomor'],
        'segment': process['segment'],
        'pipe_diameter': process['diameter'],
        'length': process['length'],
        'segment_code': process['code'],
        'normalized_acvg_dcvg_file': process['acvg_dcvg'].results[0]['excel'] if process['acvg_dcvg'] is not None else None,
        'normalized_cips_file': process['cips'].results[0]['excel'] if process['cips'].results[0]['success'] else None,
        'normalized_pcm_file': process['pcm'].results[0]['excel'] if process['pcm'].results[0]['success'] else None,
    }

    results.append(_result)

D:\Projects\corrosions\.venv\Lib\site-packages\numpy\lib\_polynomial_impl.py:674: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


In [11]:
check = pd.DataFrame(results)

### Save `labels.txt` to be used in `sync.ipynb`

In [12]:
check.to_excel(r'D:\Projects\corrosions\tests\labels.xlsx', index=False)

In [13]:
check.to_json(r'D:\Projects\corrosions\tests\labels.xlsx', index=False)

,id,year,area,area_code,nomor,segment,pipe_diameter,length,segment_code,normalized_acvg_dcvg_file,normalized_cips_file,normalized_pcm_file
0,0,2025,Jakarta,jakarta-2025,1,Pipa Servis Indonesia Power,16,1.75,pipa-servis-indonesia-power-16,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
1,1,2025,Jakarta,jakarta-2025,2,RE Martadinata - Jl. Industri Salim Ivomas 2,16,1.70,re-martadinata-jl-industri-salim-ivomas-2-16,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
2,2,2025,Jakarta,jakarta-2025,3,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,10,3.67,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
3,3,2025,Jakarta,jakarta-2025,4,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,10,0.84,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
4,4,2025,Jakarta,jakarta-2025,5,Parang Tritis - Ancol,10,1.51,parang-tritis-ancol-10,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
...,...,...,...,...,...,...,...,...,...,...,...,...
90,90,2024,Cirebon,cirebon-2024,NaN,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,6,5.32,std-garawangi-sungai-cipetir-selatan-bv-cilump...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
91,91,2024,Cilegon,cilegon-2024,NaN,Cilegon - Merak (SV 06 Grogol - SV 07),16,4.94,cilegon-merak-sv-06-grogol-sv-07-16,None,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
92,92,2024,Cilegon,cilegon-2024,NaN,Bojonegara - Suralaya (SV 01 - SV 04),16,16.53,bojonegara-suralaya-sv-01-sv-04-16,None,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
93,93,2024,Cilegon,cilegon-2024,NaN,Cilegon - Anyer (SV 05 - SV 14),16,13.30,cilegon-anyer-sv-05-sv-14-16,None,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
